# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [561]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [562]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [563]:
# GLOBAL VARIABLES
threshold = 15

In [564]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)

    dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(target_qf)
    im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
    im.qt[0] = FD.custom_q_mat(100)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}.jpeg"
    im.write_dct(output_path)

In [565]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [566]:
def get_compress_coeff(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    num_v_blocks, num_h_blocks, _, _ = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [567]:
def convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100):
    with Image.open(tiff_path) as img:
        rgb_img = img.convert('L')
        rgb_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
    print(f"Converted {tiff_path} to {jpeg_path} with quality {quality}")

In [568]:
def convert_data_to_bits(data):
    data = data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in data)
    lendata = len(data_bin)
    return data_bin, lendata

In [569]:
def sort_smoothness(smoothness_list):
    smoothness_list.sort(key=lambda x: (-x[1], x[2]))
    return smoothness_list

In [570]:
def block_smoothness_estimation(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    smoothness_block = []
    total_ec = 0
    total_zero_count = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            print( block)
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients 
            zero_count = np.sum(block_1d == 0)
            non_zero_sum = np.sum(abs(block_1d[block_1d != 0]))
            non_zero_indices = np.nonzero(block_1d)[0]
            capable_bits = 4 if zero_count == 0 else 3
            total_ec += len(non_zero_indices) // 2 * capable_bits
            total_zero_count += zero_count
            smoothness_block.append(((i, j), zero_count, non_zero_sum))

    print(f"Total blocks: {h * w}")
    print(f"Total embedding capacity (estimated): {total_ec} bits")
    print(f"Total zero count: {total_zero_count}")
    # return smoothness_block

def block_smoothness(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    q_table = im.qt[0]
    smoothness_block = []
    smoothness_score = []
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            ac_block = block.copy()
            ac_block[0, 0] = 0
            z_k = np.sum(ac_block == 0)
            E_k = np.sum((ac_block != 0) * (q_table ** 2))
            S_k = z_k + float(z_k / E_k)
            smoothness_block.append(((i, j), z_k, E_k, S_k))
            smoothness_score.append(((i, j), S_k))
    
    print(f"Total blocks: {h * w}")
    print(smoothness_block)
    print(smoothness_score)
    return smoothness_block, smoothness_score

In [571]:
def causal_neighboor_smoothness(image_path):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(1, len(coeffs)):
        sum_z_k = np.sum(coeffs[idx-1][1:] == 0)
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    mean_ac = np.mean([block[2] for block in smoothness_block])
    return smoothness_block, int(mean_ac)

In [572]:
def invariant_ac_smoothness(image_path, threshold_eob):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(len(coeffs)):
        sum_z_k = 0
        sum_ac_k = 0
        for k in range(threshold_eob + 1, 64):
            if coeffs[idx][k] == 0:
                sum_z_k += 1
            sum_ac_k += abs(coeffs[idx][k])
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    s_smoothness_block = sort_smoothness(smoothness_block)
    return smoothness_block, s_smoothness_block

In [573]:
def optimal_zero_pair_selection(image_path, threshold, payload=None, s_block=None, t_smooth=None):
    coeffs = get_quantized_coefficients(image_path) 
    num_pairs = (len(coeffs[0]) - 1) // 2
    bits_at_position = [0] * num_pairs 
    zero_pair_counts = [0] * num_pairs 
    t_select = 0
    total_ec_global = 0 

    for idx in range(1, len(coeffs)):
        N = 4 if (s_block[idx - 1][2] > t_smooth) else 3
        idx_pairs = 0
        for k in range(2, 64, 2): 
            if idx_pairs >= threshold: break
            e1 = coeffs[idx][k-1] - coeffs[idx - 1][k-1]
            e2 = coeffs[idx][k] - coeffs[idx - 1][k]
            if e1 == 0 and e2 == 0:
                bits_at_position[idx_pairs] += N
                zero_pair_counts[idx_pairs] += 1
                total_ec_global += N
            idx_pairs += 1

    if payload:
        if total_ec_global < payload:
            raise ValueError(f"Warning: Kapasitas total ({total_ec_global}) tidak mencukupi payload ({payload})")    
        else:
            current_accumulated_bits = 0
            for i in range(len(bits_at_position)):
                current_accumulated_bits += bits_at_position[i]
                if current_accumulated_bits >= payload:
                    t_select = i + 1 
                    break

    print(f"Statistik Zero Pairs per Posisi Zigzag: {zero_pair_counts}")
    print(f"Kapasitas per posisi zigzag (bits): {bits_at_position}")
    print(f"Optimal Threshold T yang dipilih: {t_select}")
    print(f"Total Kapasitas tersedia: {total_ec_global} bits")
    return t_select, total_ec_global, zero_pair_counts

In [574]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

def get_nacp_2(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) > 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [575]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

def replace_nacp_2(sorted_coefficients, nacp_coords):
    pair_index = 0
    for zigzag_coeff in sorted_coefficients:
        ac_part = zigzag_coeff[1:]
        rel_non_zero_indices = np.nonzero(np.abs(ac_part) > 1)[0]        
        for idx in range(0, len(rel_non_zero_indices) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]                
                zigzag_coeff[rel_non_zero_indices[idx] + 1] = float(new_x)
                zigzag_coeff[rel_non_zero_indices[idx+1] + 1] = float(new_y)
                pair_index += 1
            else: break
    return sorted_coefficients

In [576]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

def construct_stego_file_2(image_path, new_coeffs, qf):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    if qf is not None:    
        dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(qf)
        im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
        im.qt[0] = FD.custom_q_mat(100)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [577]:
def recovered_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "recovered-images/recovered_" + image_path.split("/")[-1]
    print(f"Recovered image saved to {output_path}")
    im.write_dct(output_path)

In [578]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    if secret_data == "": return nacp_coord
    bit = 3 if mode == "8N" else 4
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), bit):
        group = data_bin[i:i+bit].ljust(bit, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = turtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if turtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            # shell_coords = turtleShell.get_kxk_nearest_signed(x, y, bit)
            _, shell_coords = turtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = turtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [579]:
def data_extract_process(nacp_coord, mode="8N"):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
        bit = 3 if mode == "8N" else 4
        bits = format(val & ((1 << bit) - 1), f'0{bit}b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [580]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

def encode_2(image_path, data, qf):
    image = Image.open(image_path).convert('L')
    stegoimg = image.copy()
    img_arr = np.array(stegoimg)
    q_mat = FD.custom_q_mat(qf)
    sorted_coefficients = FD.transform_to_freq(img_arr, q_mat)
    nacp_coords = get_nacp(sorted_coefficients)
    modified_nacp_coords = data_hiding_process(data, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coefficients, modified_nacp_coords)
    np.save("modified_coefficients.npy", modified_coeffs)

def encode_4(image_path, secret_data, qf):
    sorted_coeffs = get_compress_coeff(image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(secret_data, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file_2(image_path, modified_coeffs, qf)
    
# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def encode_3(image_path, secret_data):
    _, smoothness_score = block_smoothness(image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    secret_data += '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)

    _, shells, cell_to_shells = turtleShell.init(mode="8N")
    _, shells_17, cell_to_shells_17 = turtleShell.init(mode="17N")
    
    im = jpeglib.read_dct(image_path)
    for (block_i, block_j), score in smoothness_list:
        # print(f"Processing block ({block_i}, {block_j}) with smoothness score {score}")
        if lendata <= 0: break
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        choosen_shells = shells if N == 8 else shells_17
        choosen_cell_to_shells = cell_to_shells if N == 8 else cell_to_shells_17
        for idx in range(0, len(non_zero_indices) - 1, 2):
            if lendata <= 0: break
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            bits = data_bin[:t].ljust(t, '0') 
            data_bin = data_bin[t:]
            lendata -= t
            target_val = int(bits, 2)
            val_int = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            if target_val != val_int:
                _, shell_coords = turtleShell.get_shell_coords(x, y, choosen_shells, choosen_cell_to_shells)
                x, y = turtleShell.find_corresponding_val(shell_coords, target_val, (x, y), mode=mode)
            ac_coeffs[non_zero_indices[idx]] = float(x)
            ac_coeffs[non_zero_indices[idx + 1]] = float(y)

        zigzag_coeffs[1:] = ac_coeffs
        zigzag_coeffs[0] = block[0, 0]  
        im.Y[block_i, block_j] = inverse_zigzag(zigzag_coeffs, 8, 8)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Data embedding completed. Stego image saved to {output_path}")
    im.write_dct(output_path)

In [ ]:
def rdh_embed_process(image_path, data_bin, t_select, s_block, t_smooth):
    coeffs = get_quantized_coefficients(image_path) 
    orig_coeffs = copy.deepcopy(coeffs) 
    data_idx = 0
    lendata = len(data_bin)
    total_embed = 0
    total_shift = 0

    for idx in range(1, len(coeffs)):
        if data_idx >= lendata: break 
        sum_ac_prev = np.sum(np.abs(orig_coeffs[idx-1][1:]))
        N = 4 if (sum_ac_prev < t_smooth) else 3
        mode = "8N" if N == 3 else "17N"
        radius = 1 if N == 3 else 2         
        for k in range(2, t_select * 2 + 1, 2):
            if data_idx >= lendata: break 
            e1 = int(orig_coeffs[idx][k-1] - orig_coeffs[idx - 1][k-1])
            e2 = int(orig_coeffs[idx][k] - orig_coeffs[idx - 1][k])
            if e1 == 0 and e2 == 0:
                bits = data_bin[data_idx : data_idx + N].ljust(N, '0')
                data_idx += N
                total_embed += 1                
                target_val = int(bits, 2)
                candidate_coords = turtleShell.get_kxk_nearest_zero(0, 0, N)
                e1_mod, e2_mod = turtleShell.find_val_from_zero(candidate_coords, target_val, (e1, e2), mode=mode)
                print(f"Embedding bits: {bits} as {target_val} in ({e1_mod}, {e2_mod}) from ({e1}, {e2})")
            else:
                if not (e1 == 0 and e2 == 0):
                    total_shift += 1
                    e1_mod = e1 + int(np.sign(e1)) * radius if e1 != 0 else 0
                    e2_mod = e2 + int(np.sign(e2)) * radius if e2 != 0 else 0
                else:
                    e1_mod, e2_mod = e1, e2
            coeffs[idx][k-1] = orig_coeffs[idx - 1][k-1] + e1_mod
            coeffs[idx][k] = orig_coeffs[idx - 1][k] + e2_mod

    print(f"DEBUG: Berhasil Embed: {total_embed} pasang, Berhasil Shift: {total_shift} pasang")
    print(f"DEBUG: Sisa data bit: {max(0, lendata - data_idx)}")
    construct_stego_file(image_path, coeffs)
    return coeffs

In [582]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_encode(image_path, secret_data):
    ori_coeff = get_quantized_coefficients(image_path)
    data_bin, lendata = convert_data_to_bits(secret_data)
    print(secret_data)
    print(f"Data bits: {data_bin}")
    print(f"Data bits length: {lendata}")    
    s_block, mean_thresold = causal_neighboor_smoothness(image_path)
    t_select, _, _ = optimal_zero_pair_selection(image_path, 21, payload=len(secret_data)*8, s_block=s_block, t_smooth=mean_thresold)
    coeff_after_embed = rdh_embed_process(image_path, data_bin, t_select, s_block, mean_thresold)
    diff = np.array(ori_coeff) - np.array(coeff_after_embed)
    return diff, t_select, mean_thresold

In [583]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

def decode_2(stego_file):
    modified_coeffs = np.load(stego_file, allow_pickle=True)
    nacp_coords = get_nacp(modified_coeffs)
    extracted_data = data_extract_process(nacp_coords)
    return extracted_data

def decode_4(stego_image_path, qf):
    sorted_coeffs = get_compress_coeff(stego_image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def decode_3(stego_image_path):
    _, smoothness_score = block_smoothness(stego_image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    im = jpeglib.read_dct(stego_image_path)
    bitstream = ""
    decoded_text = ""
    for (block_i, block_j), score in smoothness_list:
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]

        for idx in range(0, len(non_zero_indices) - 1, 2):
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            bits = format(val, f"0{t}b")
            bitstream += bits
            while len(bitstream) >= 8:
                byte = bitstream[:8]
                bitstream = bitstream[8:]
                char_val = int(byte, 2)
                if char_val == 0:   # Null terminator
                    return decoded_text
                decoded_text += chr(char_val)
    return decoded_text

In [584]:
def rdh_extract_process(stego_image_path, t_select, t_smooth):
    coeffs = get_quantized_coefficients(stego_image_path) 
    bit_stream = ""    
    secret_data = "" 
    stop_extraction = False

    for idx in range(1, len(coeffs)):
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        N = 4 if (sum_ac_k < t_smooth) else 3
        mode = "8N" if N == 3 else "17N" 
        radius = 1 if N == 3 else 2 
        for k in range(2, t_select * 2 + 1, 2): 
            if stop_extraction: break
            d1 = int(coeffs[idx][k-1] - coeffs[idx - 1][k-1])
            d2 = int(coeffs[idx][k] - coeffs[idx - 1][k])
            if turtleShell.is_in_central_shell(d1, d2, mode=mode):
                if not stop_extraction:
                    val = turtleShell.get_zero_matrix_value(d1, d2, mode=mode)
                    bit_stream += format(val, f'0{N}b')
                    while len(bit_stream) >= 8:
                        byte = bit_stream[:8]
                        bit_stream = bit_stream[8:]
                        char_val = int(byte, 2)
                        if char_val == 0: 
                            stop_extraction = True
                            break
                        secret_data += chr(char_val)
                e1, e2 = 0, 0
            else:
                e1 = d1 - int(np.sign(d1)) * radius if d1 != 0 else 0
                e2 = d2 - int(np.sign(d2)) * radius if d2 != 0 else 0
            coeffs[idx][k-1] = coeffs[idx - 1][k-1] + e1
            coeffs[idx][k] = coeffs[idx - 1][k] + e2
    recovered_stego_file(stego_image_path, coeffs)
    return secret_data

In [585]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_decode(stego_image_path, t_select = 21, t_smooth=2):
    secret_data = rdh_extract_process(stego_image_path, t_select=t_select, t_smooth=t_smooth)
    return secret_data

In [586]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [587]:
convert_tiff_to_jpeg("cover-images/misc/boat.512.tiff", "cover-images/boat_qf50.jpeg", quality=50)

Converted cover-images/misc/boat.512.tiff to cover-images/boat_qf50.jpeg with quality 50


In [588]:
pay_size = 100
# cover_folder = "cover-images/"
# stego_folder = "stego-images/"
payload_folder = "payload/"
# cover_image_path = f"boat_qf70.jpeg"
# stego_image_path = f"stego_boat_qf70.jpeg"
# # data = "1"
# data = read_text_file(f"{payload_folder}{pay_size}Kb.txt")
# # encode_4(f"{cover_folder}{cover_image_path}", data, 50)
# encode(f"{cover_folder}{cover_image_path}", data)

# secret_data = decode(f"{stego_folder}{stego_image_path}")
# # secret_data = decode_4(f"{stego_folder}{stego_image_path}", 50)
# print("Extracted Data:", secret_data) 

# data = "Hi"
data = read_text_file(f"{payload_folder}{pay_size}Kb.txt")
image = "cover-images/baboon_qf50.jpeg"
diff, t_select, mean_thresold = rdh_encode(image, data)
print("Difference Coefficients:")
for i in range(len(diff)):
    print(diff[i])
print(f"Difference Length: {len(diff)}")

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi sed mollis sem. Aenean imperdiet risus vitae lectus consectetur feugiat. Sed sit amet purus eu ex consectetur sollicitudin at et erat. Pellentesque vel finibus sem. Etiam eu tempus massa. Quisque ultricies aliquet nisl, vel vulputate turpis ornare eu. Integer vulputate quam a lacinia dapibus. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Vestibulum suscipit egestas faucibus. Maecenas venenatis ultrices dolor in blandit. Duis auctor, neque a aliquet vehicula, eros quam cursus orci, in feugiat eros purus viverra velit.
Duis ante justo, molestie ut enim ut, vulputate ultricies sem. Vivamus faucibus metus quam, vel varius est ornare sit amet. Duis efficitur, nulla id dictum elementum, tellus libero efficitur risus, at commodo dolor justo ac augue. Ut hendrerit, neque non interdum viverra, est lacus faucibus elit, eu interdum ligula est vel ante. Cras vitae nisi justo. Nulla facilisi

In [589]:
secret_data = rdh_decode("stego-images/stego_baboon_qf50.jpeg", t_select=t_select, t_smooth=mean_thresold)
print("Extracted Data:", secret_data)

Recovered image saved to recovered-images/recovered_stego_baboon_qf50.jpeg
Extracted Data: Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi sed mollis sem. Aenean imperdiet risus vitae lectus consectetur feugiat. Sed sit amet purus eu ex consectetur sollicitudin at et erat. Pellentesque vel finibus sem. Etiam eu tempus massa. Quisque ultricies aliquet nisl, vel vulputate turpis ornare eu. Integer vulputate quam a lacinia dapibus. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Vestibulum suscipit egestas faucibus. Maecenas venenatis ultrices dolor in blandit. Duis auctor, neque a aliquet vehicula, eros quam cursus orci, in feugiat eros purus viverra velit.
Duis ante justo, molestie ut enim ut, vulputate ultricies sem. Vivamus faucibus metus quam, vel varius est ornare sit amet. Duis efficitur, nulla id dictum elementum, tellus libero efficitur risus, at commodo dolor justo ac augue. Ut hendrerit, neque non interdum viverra, est 

In [590]:
# Test performance metrics
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"stego_baboon_qf50.jpeg"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 45747
Size stego: 73713
PSNR: 17.923254121582808 dB
FSI: 27966.0
SSIM: 0.5359921881420547


In [591]:
# Test performance metrics
cover_folder = "cover-images/"
recovered_folder = "recovered-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"recovered_stego_baboon_qf50.jpeg"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 45747
Size stego: 45783
PSNR: inf dB
FSI: 36.0
SSIM: 1.0


In [592]:
def compare_spatial_frequency(cover_image_path, stego_image_path):
    cover = np.array(Image.open(cover_image_path).convert('L'), dtype=np.float64)
    stego = np.array(Image.open(stego_image_path).convert('L'), dtype=np.float64)

    spatial_difference = np.abs(cover - stego) ** 2

    cover_freq = np.fft.fft2(cover)
    stego_freq = np.fft.fft2(stego)
    freq_difference = np.abs(cover_freq - stego_freq)

    return freq_difference, spatial_difference

In [593]:
# freq_diff, spatial_diff = compare_spatial_frequency(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
# print("Frequency Difference:")
# for row in freq_diff:
#     print(row)
# print("Spatial Difference:")
# for row in spatial_diff:
#     print(row)
# print("Max spatial diff:", spatial_diff.max())
# print("Mean spatial diff:", spatial_diff.mean())

In [594]:
# max_pixel = 255.0
# psnr_value = 20 * log10(max_pixel / sqrt(spatial_diff.mean()))
# print("PSNR calculated from spatial difference:", psnr_value, "dB")

In [595]:
# Test performance metrics
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"boat_qf50.jpeg"
stego_image_path = f"stego_boat_qf50.jpeg"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 27024
Size stego: 28736
PSNR: 29.953026424384028 dB
FSI: 1712.0
SSIM: 0.8058871023685921
